In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## **Function 1** - Searching for contamination sources

For this function we must detect likely contamination sources in a two-dimensional area, such as a radiation field, where only proximity yields a non-zero reading.

- The system uses Bayesian optimisation to tune detection parameters and reliably identify both strong and weak sources.

- **Optimisation Goal** - Maximise

I want to begin this week's analysis by loading the data and appending new week's data to last week's updated data.

In [2]:
X = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_1/initial_inputs.npy')
Y = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_1/initial_outputs.npy')

# New data for Week 6 (Function 1)
X_w5_new_point = np.array([0.472361, 0.351758], dtype=np.float64)
Y_w5_new_point = np.array([0.000002804634603937215], dtype=np.float64)

# Append the new data point
X_updated = np.vstack((X, X_w5_new_point.reshape(1, -1)))

# Remove duplicate X rows (if point already exists)
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)

# Build Y and keep matching rows only
Y_all = np.append(Y, Y_w5_new_point)
Y_updated = Y_all[unique_indices]

# Save the updated arrays
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_1/initial_inputs.npy', X_unique)
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_1/initial_outputs.npy', Y_updated)

# Show updated arrays
print("Updated Inputs (X) - Function 1: ", X_unique)
print("Updated Outputs (Y) - Function 1: ", Y_updated)

Updated Inputs (X) - Function 1:  [[0.08250725 0.40348751]
 [0.31269116 0.07872278]
 [0.31940389 0.76295937]
 [0.41043714 0.1475543 ]
 [0.472361   0.351758  ]
 [0.57432921 0.8798981 ]
 [0.575251   0.879599  ]
 [0.65011406 0.68152635]
 [0.68341817 0.86105746]
 [0.73102363 0.73299988]
 [0.732441   0.040134  ]
 [0.84035342 0.26473161]
 [0.88388983 0.58225397]
 [0.919191   0.838383  ]
 [0.996651   0.001439  ]]
Updated Outputs (Y) - Function 1:  [ 3.60677119e-081 -2.08909327e-091  1.32267704e-079 -2.15924904e-054
  2.80463460e-006  1.03307824e-046  1.46776061e-046 -3.60606264e-003
  2.53500115e-040  7.71087511e-016 -5.11184814e-169  3.34177101e-124
  6.22985647e-048 -3.04056455e-090  0.00000000e+000]


## Output Analysis
- Last week (W4) we saw $-5.11 \times 10^{-169}$, and this week (W5) we saw $2.80 \times 10^{-06}$.

- **Real-World Implication**: This is a monumental shift. I have moved from a value that is physically indistinguishable from zero (W4) to a value that, while small, is many orders of magnitude larger. 
    - I have finally "pinged" the sensor. 
    - I have transitioned from absolute silence to the very first detectable signal of the contamination plume.

- The "needle in a haystack" has finally been nudged. 
    - This confirms that the search space is mostly an empty floor, but we have found the edge of the signal. 
    - I must now determine if this is a minor outlier or the start of the gradient leading to the primary source.

### **Bayesian Optimisation**: Gaussian Process Regressor

- **Model Understanding**: We am using a Matern kernel with nu=2.5 to handle the spatial distribution of the radiation, paired with a WhiteKernel to manage the noise.
- **Changes/Reasons**: I am keeping the model consistent. 
    - I just achieved our first detectable result in the history of the project; changing the model architecture now would be statistically reckless. 
    - I need the GP to process this new anchor point using the same assumptions to see if a clear peak begins to emerge.

- To allow the GP to stabilize and begin forming a localized gradient around the W5 coordinates, providing a search direction for the first time.

In [3]:
# Define the model - Maintaining Matern 5/2 for spatial field modeling
kernel = Matern(
    length_scale=1.0, 
    length_scale_bounds=(1e-2, 1e2), 
    nu=2.5) + \
         WhiteKernel(noise_level=1e-5)

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=10,
    normalize_y=True)

model.fit(X_unique, Y_updated)

/Users/prathamyeole/Library/Python/3.13/lib/python/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",Matern(length...e_level=1e-05)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",10
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`G

### **Acquisition Function**

- I am using Upper Confidence Bound (UCB) with $\beta=5.0$.

- **Changes/Reasons**: I am maintaining $\beta=5.0$. 
    - Although I found a signal, $2.80 \times 10^{-06}$ is still far too low to be the global maximum. 
    - Lowering $\beta$ would lead to premature exploitation of a very weak signal. 
        - I must remain in high-exploration mode to see if this signal leads to a significantly higher peak nearby or if there is a better source elsewhere.

- To prioritize the largest uncertainty gaps while being slightly influenced by the new non-zero coordinate.

In [4]:
def upper_confidence_bound(X, model, beta=5.0):
    mu, sigma = model.predict(X, return_std=True)
    return mu + beta * sigma

# Generate a high-resolution grid for the 2D space
x_grid = np.array(np.meshgrid(np.linspace(0, 1, 250), np.linspace(0, 1, 250))).T.reshape(-1, 2)

# Calculate UCB values using Beta=5.0
ucb_values = upper_confidence_bound(x_grid, model, beta=5.0)

# Select the next query
next_query = x_grid[np.argmax(ucb_values)]

print(f"Strategic Week 6 Query (Function 1): [{next_query[0]:.6f}-{next_query[1]:.6f}]")

Strategic Week 6 Query (Function 1): [0.799197-0.803213]
